In [1]:
import tarfile
import gzip
import os
import json
import pickle
import urllib.parse
import re

In [2]:
import os
import gzip
import json
os.chdir("D:\Entity Aspect Linking\data\entity-aspect-linking-2020\collection")
path = "test.jsonl.gz"
data = {}
i = 1
with gzip.open(path, 'rt', encoding='UTF-8') as zipfile:
    for line in zipfile:
        my_object = json.loads(line)
        data[str(i)] = my_object
        i = i + 1

In [3]:
def decode(s):
    s = s.replace("enwiki:","")
    return urllib.parse.unquote(s)

In [4]:
def clean_aspect(s):
    s = decode(s)
    pattern = re.compile(r'.*?/')
    return re.sub(pattern, '', s)

In [6]:
def make_entdict(data, n = 10, israndom = False, ind = []):
    ent_data = []
    if israndom == True:
        it = list(map(int, ind))
    else:
        it = range(n)
    i = 0
    for i in it:
        temp = {}
        ent = data[str(i+1)]
        temp["o_id"] = ent["id"]
        temp["id"] = str(i)
        temp["target_entity"] = decode(ent['context']['target_entity'])
        temp["sentence"] = ent['context']['sentence']['content']
        temp["entities"] = []                        
        k = 0
        for j in ent['context']['sentence']['entities']:
            ent = {}                          
            if not j["target_mention"]:
                ent['eid'] = str(i) + str(k)
                ent['entity'] = j['entity_name']
                ent['mention'] = j['mention']
                temp["entities"].append(ent)
            k += 1
        ent_data.append(temp)

    return ent_data

In [7]:
ent = make_entdict(data, n = len(data.keys()))

In [8]:
def get_aspectdict(data, n = 10, israndom = False, ind = []):
    aspects = []
    k = 0
    if israndom == True:
        it = list(map(int, ind))
    else:
        it = range(n)
    for i in it:
        temp = {}
        ent = data[str(i + 1)]
        temp['id'] = str(i)
        temp['true_aspect_id'] = ent['true_aspect']
        temp['true_aspect'] = clean_aspect(ent['true_aspect'])
        candasp = []
        j = 0
        for cand in ent['candidate_aspects']:
            casp = {}
            casp['aspect_id'] = cand['aspect_id']
            casp['id'] = 'A' + str(i) + str(j)
            casp['aspect_name'] = cand['aspect_name']
            casp['section_heading'] = cand['location']['section_headings']
            casp['content'] = cand['aspect_content']['content']
            ent = []
            for e in cand['aspect_content']['entities']:
                temp2 = {}
                temp2['entity_name'] = e['entity_name']
                temp2['eid'] = 'E' + str(k)
                temp2['mention'] = e['mention']
                ent.append(temp2)
                k+= 1
            casp['entities'] = ent
            candasp.append(casp)
            j += 1
        temp['candidate_aspects'] = candasp
        aspects.append(temp)
    return aspects

In [9]:
asp = get_aspectdict(data, n = len(data.keys()))

In [10]:
os.chdir("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\Sentence\\picklefiles")
PTH = os.getcwd()
final = []
for i in range(len(ent)):
    final.append((ent[i], asp[i]))

In [11]:
with open(f"{PTH}\\eal_test.pkl", 'wb') as f:
    pickle.dump(final, f)
print("Dumped")
f.close()

Dumped
